# ResNet18 Training & Evaluation (Stratified 80/20)
Tämä notebook kouluttaa ResNet18-mallin `ImageFolder`-datalle stratifioidulla 80/20-jaolla ja sisältää oikeat eval-laskennat sekä 20 kuvan pikacheckin (ennuste + oikea luokka).

In [1]:

# %% [markdown]
# ## Imports & peruskonfiguraatio

import os, json, random, numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedShuffleSplit

from PIL import Image

# Laitteisto
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Polut (MUOKKAA images_dir omaan dataasi sopivaksi)
# images_dir: polku juurikansioon, jossa on alikansiot per luokka (ImageFolder-muoto)
images_dir = Path("images")  # esim. Path("/content/drive/MyDrive/Trees/images")

# Koulutusparametrit
config = {
    "image_size": 224,
    "batch_size": 32,
    "epochs": 10,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "seed": 1234,
    "num_workers": 2,
    "save_dir": "checkpoints",
    "model_name": "Resnet18_V3.pth"
}
os.makedirs(config["save_dir"], exist_ok=True)

# Siemenet determinismiin
random.seed(config["seed"]); np.random.seed(config["seed"]); torch.manual_seed(config["seed"])
if device.type == "cuda":
    torch.cuda.manual_seed_all(config["seed"])

print("Config:", json.dumps(config, indent=2))
print("PyTorch:", torch.__version__)


Device: cpu
Config: {
  "image_size": 224,
  "batch_size": 32,
  "epochs": 10,
  "lr": 0.001,
  "weight_decay": 0.0001,
  "seed": 1234,
  "num_workers": 2,
  "save_dir": "checkpoints",
  "model_name": "Resnet18_V3.pth"
}
PyTorch: 2.5.1


In [2]:

# %% [markdown]
# ## Transforms ja stratifioitu 80/20 -jako

# Train-transform (voisit lisätä augmentaatioita tarvittaessa)
train_tf = transforms.Compose([
    transforms.Resize((config["image_size"], config["image_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# Val-transform (ei augmentaatioita)
val_tf = transforms.Compose([
    transforms.Resize((config["image_size"], config["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# Lataa koko data KERRAN
full = datasets.ImageFolder(images_dir, transform=None)
print("Classes:", full.classes)
print("class_to_idx:", full.class_to_idx)
print("Kuvia yhteensä:", len(full.samples))

# Stratifioitu jako luokkien mukaan (80/20)
y = [lbl for _, lbl in full.samples]
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=config["seed"])
train_idx, val_idx = next(sss.split(np.zeros(len(y)), y))

assert set(train_idx).isdisjoint(set(val_idx)), "Train/Val -indeksit menevät päällekkäin!"

# Luo Subsetit samaan taustadataan (class_to_idx pysyy identtisenä)
train_data = Subset(datasets.ImageFolder(images_dir, transform=train_tf), train_idx)
val_data   = Subset(datasets.ImageFolder(images_dir, transform=val_tf),   val_idx)

print("Train n:", len(train_idx), "Val n:", len(val_idx))

# Dataloaderit
train_loader = DataLoader(train_data, batch_size=config["batch_size"], shuffle=True,
                          num_workers=config["num_workers"], pin_memory=True)
val_loader   = DataLoader(val_data,   batch_size=config["batch_size"], shuffle=False,
                          num_workers=config["num_workers"], pin_memory=True)

# Indeksistä luokan nimeen -map (talteen inferenssiin)
idx_to_class = {v:k for k,v in full.class_to_idx.items()}
with open(Path(config["save_dir"]) / "class_to_idx.json", "w") as f:
    json.dump(full.class_to_idx, f, indent=2)
print("Tallennettu class_to_idx -> checkpoints/class_to_idx.json")


Classes: ['koivu', 'kuusi', 'lehmus', 'pihlaja', 'vaahtera']
class_to_idx: {'koivu': 0, 'kuusi': 1, 'lehmus': 2, 'pihlaja': 3, 'vaahtera': 4}
Kuvia yhteensä: 136
Train n: 108 Val n: 28
Tallennettu class_to_idx -> checkpoints/class_to_idx.json


In [3]:

# %% [markdown]
# ## ResNet18-malli

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(full.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])

print("Model ready. Classes:", len(full.classes))


Model ready. Classes: 5


In [4]:

# %% [markdown]
# ## Treeni- ja validaatiofunktiot (oikea keskiarvoistus)

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            total_loss += loss.item() * y.size(0)
            preds = logits.argmax(1)
            total_correct += (preds == y).sum().item()
            total_samples += y.size(0)

    return total_loss/total_samples, 100.0 * total_correct/total_samples


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y, reduction='sum')
        total_loss += loss.item()
        preds = logits.argmax(1)
        total_correct += (preds == y).sum().item()
        total_samples += y.size(0)

    return total_loss/total_samples, 100.0 * total_correct/total_samples


In [5]:

# %% [markdown]
# ## Koulutus

best_val_acc = -1.0
save_path = Path(config["save_dir"]) / config["model_name"]

print("Starting model training...")
for epoch in range(1, config["epochs"]+1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc     = evaluate(model, val_loader, device)
    scheduler.step()

    print(f"Epoch {epoch}/{config['epochs']} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    # Tallenna paras malli val-accin mukaan
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        with open(Path(config["save_dir"])/"config.json", "w") as f:
            json.dump(config, f, indent=2)
        print(f"  -> Saved best checkpoint to {save_path} (Val Acc: {best_val_acc:.2f}%)")

print("Training finished.")


Starting model training...
Epoch 1/10 | Train Loss: 0.7532 | Train Acc: 72.22% | Val Loss: 1.2239 | Val Acc: 60.71%
  -> Saved best checkpoint to checkpoints/Resnet18_V3.pth (Val Acc: 60.71%)
Epoch 2/10 | Train Loss: 0.0696 | Train Acc: 98.15% | Val Loss: 9.3473 | Val Acc: 17.86%
Epoch 3/10 | Train Loss: 0.0775 | Train Acc: 98.15% | Val Loss: 22.7951 | Val Acc: 21.43%
Epoch 4/10 | Train Loss: 0.0282 | Train Acc: 99.07% | Val Loss: 14.5580 | Val Acc: 35.71%
Epoch 5/10 | Train Loss: 0.0954 | Train Acc: 97.22% | Val Loss: 1.5372 | Val Acc: 78.57%
  -> Saved best checkpoint to checkpoints/Resnet18_V3.pth (Val Acc: 78.57%)
Epoch 6/10 | Train Loss: 0.0220 | Train Acc: 99.07% | Val Loss: 0.5037 | Val Acc: 85.71%
  -> Saved best checkpoint to checkpoints/Resnet18_V3.pth (Val Acc: 85.71%)
Epoch 7/10 | Train Loss: 0.0093 | Train Acc: 100.00% | Val Loss: 0.3171 | Val Acc: 89.29%
  -> Saved best checkpoint to checkpoints/Resnet18_V3.pth (Val Acc: 89.29%)
Epoch 8/10 | Train Loss: 0.1047 | Train Acc

In [7]:

# %% [markdown]
# ## Predict: yksittäinen kuva

@torch.no_grad()
def predict(pil_image, model, device):
    model.eval()
    tf = transforms.Compose([
        transforms.Resize((config["image_size"], config["image_size"])),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    x = tf(pil_image).unsqueeze(0).to(device)
    logits = model(x)
    prob = torch.softmax(logits, dim=1)[0].cpu().numpy()
    pred_idx = int(prob.argmax())
    return {
        "pred_class": idx_to_class[pred_idx],
        "probs": {idx_to_class[i]: float(prob[i]) for i in range(len(prob))}
    }

# Esimerkki (kommentoi päälle ja anna oma polku)
# img = Image.open('path/to/your/image.jpg').convert('RGB')
# out = predict(img, model, device)
# out
